In [ ]:
from google.colab import files
uploaded = files.upload()

Saving flipkart_products_price.zip to flipkart_products_price.zip


In [ ]:
import pandas as pd

df = pd.read_csv("flipkart_products_price.zip")
df.head()

,uniq_id,crawl_timestamp,product_url,product_name,product_category_tree,pid,retail_price,discounted_price,image,is_FK_Advantage_product,description,product_rating,overall_rating,brand,product_specifications
0,c2d766ca982eca8304150849735ffef9,2016-03-25 22:59:23 +0000,http://www.flipkart.com/alisha-solid-women-s-c...,Alisha Solid Women's Cycling Shorts,"[""Clothing >> Women's Clothing >> Lingerie, Sl...",SRTEH2FF9KEDEFGF,999.0,379.0,"[""http://img5a.flixcart.com/image/short/u/4/a/...",False,Key Features of Alisha Solid Women's Cycling S...,No rating available,No rating available,Alisha,"{""product_specification""=>[{""key""=>""Number of ..."
1,7f7036a6d550aaa89d34c77bd39a5e48,2016-03-25 22:59:23 +0000,http://www.flipkart.com/fabhomedecor-fabric-do...,FabHomeDecor Fabric Double Sofa Bed,"[""Furniture >> Living Room Furniture >> Sofa B...",SBEEH3QGU7MFYJFY,32157.0,22646.0,"[""http://img6a.flixcart.com/image/sofa-bed/j/f...",False,FabHomeDecor Fabric Double Sofa Bed (Finish Co...,No rating available,No rating available,FabHomeDecor,"{""product_specification""=>[{""key""=>""Installati..."
2,f449ec65dcbc041b6ae5e6a32717d01b,2016-03-25 22:59:23 +0000,http://www.flipkart.com/aw-bellies/p/itmeh4grg...,AW Bellies,"[""Footwear >> Women's Footwear >> Ballerinas >...",SHOEH4GRSUBJGZXE,999.0,499.0,"[""http://img5a.flixcart.com/image/shoe/7/z/z/r...",False,Key Features of AW Bellies Sandals Wedges Heel...,No rating available,No rating available,AW,"{""product_specification""=>[{""key""=>""Ideal For""..."
3,0973b37acd0c664e3de26e97e5571454,2016-03-25 22:59:23 +0000,http://www.flipkart.com/alisha-solid-women-s-c...,Alisha Solid Women's Cycling Shorts,"[""Clothing >> Women's Clothing >> Lingerie, Sl...",SRTEH2F6HUZMQ6SJ,699.0,267.0,"[""http://img5a.flixcart.com/image/short/6/2/h/...",False,Key Features of Alisha Solid Women's Cycling S...,No rating available,No rating available,Alisha,"{""product_specification""=>[{""key""=>""Number of ..."
4,bc940ea42ee6bef5ac7cea3fb5cfbee7,2016-03-25 22:59:23 +0000,http://www.flipkart.com/sicons-all-purpose-arn...,Sicons All Purpose Arnica Dog Shampoo,"[""Pet Supplies >> Grooming >> Skin & Coat Care...",PSOEH3ZYDMSYARJ5,220.0,210.0,"[""http://img5a.flixcart.com/image/pet-shampoo/...",False,Specifications of Sicons All Purpose Arnica Do...,No rating available,No rating available,Sicons,"{""product_specification""=>[{""key""=>""Pet Type"",..."


In [ ]:
df = df[['product_name', 'retail_price', 'discounted_price', 'product_rating']]
df = df.dropna()
df.head()

,product_name,retail_price,discounted_price,product_rating
0,Alisha Solid Women's Cycling Shorts,999.0,379.0,No rating available
1,FabHomeDecor Fabric Double Sofa Bed,32157.0,22646.0,No rating available
2,AW Bellies,999.0,499.0,No rating available
3,Alisha Solid Women's Cycling Shorts,699.0,267.0,No rating available
4,Sicons All Purpose Arnica Dog Shampoo,220.0,210.0,No rating available


In [ ]:
df['product_rating'] = df['product_rating'].astype(str)

df['product_rating'] = df['product_rating'].replace("No rating available", "0")
df['product_rating'] = df['product_rating'].replace("No Rating Available", "0")
df['product_rating'] = df['product_rating'].replace("nan", "0")

df['product_rating'] = df['product_rating'].astype(float)

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression

X = df[['retail_price', 'product_rating']]
y = df['discounted_price']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = LinearRegression()
model.fit(X_train, y_train)

print("Model trained successfully!")

Model trained successfully!


In [ ]:
!pip install requests beautifulsoup4

In [ ]:
import requests
from bs4 import BeautifulSoup

headers = {"User-Agent": "Mozilla/5.0"}

In [ ]:
def get_amazon_price(product_name):
    search = product_name.replace(" ", "+")
    url = f"https://www.amazon.in/s?k={search}"

    r = requests.get(url, headers=headers)
    soup = BeautifulSoup(r.text, "html.parser")

    price = soup.find("span", class_="a-price-whole")

    if price:
        return int(price.text.replace(",", "").strip())
    return None

In [ ]:
i = 18

selected_product = df.iloc[i]

product_name = selected_product['product_name']
flipkart_price = selected_product['discounted_price']
retail_price = selected_product['retail_price']
rating = selected_product['product_rating']

print("Product Selected:", product_name)
print("Flipkart Price:", flipkart_price)

Product Selected: FabHomeDecor Fabric Double Sofa Bed
Flipkart Price: 22646.0


In [ ]:
amazon_price = get_amazon_price(product_name)
print("Amazon Price:", amazon_price)

Amazon Price: None


In [ ]:
data = {
    "Website": ["Flipkart", "Amazon"],
    "Price": [flipkart_price, amazon_price]
}

compare_df = pd.DataFrame(data)
compare_df

,Website,Price
0,Flipkart,22646.0
1,Amazon,NaN


In [ ]:
if amazon_price is not None:
    best = compare_df.loc[compare_df["Price"].idxmin()]
    print("\nBest Platform:", best["Website"])
    print("Best Price:", best["Price"])
else:
    print("Amazon price not found.")

Amazon price not found.


In [ ]:
future_price = model.predict([[retail_price, rating]])
future_price = int(future_price[0])

print("\nPredicted Future Price:", future_price)


Predicted Future Price: 25651


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LinearRegression was fitted with feature names
  warnings.warn(


In [ ]:
print("\n===== FINAL SUMMARY =====")
print("Product:", product_name)
print("Flipkart:", flipkart_price)
print("Amazon:", amazon_price)
print("Predicted Future Price:", future_price)

if amazon_price is None:
    print("\nAmazon price not available. Cannot compare.")
elif amazon_price < flipkart_price:
    print("\n➡ Recommendation: Buy from AMAZON (Cheapest)")
elif flipkart_price < amazon_price:
    print("\n➡ Recommendation: Buy from FLIPKART (Cheapest)")
else:
    print("\n➡ Prices are same on both.")


===== FINAL SUMMARY =====
Product: FabHomeDecor Fabric Double Sofa Bed
Flipkart: 22646.0
Amazon: None
Predicted Future Price: 25651

Amazon price not available. Cannot compare.
